This notebook trains and evaluates a XGBoost model


In [ ]:
import numpy as np
import pandas as pd
import os

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    fbeta_score,
    make_scorer,
    confusion_matrix,
)

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

import joblib


In [ ]:
#Load train and test
train_df = pd.read_pickle("../data/modeling/train_df.pkl")
test_df  = pd.read_pickle("../data/modeling/test_df.pkl")

print("Train:", train_df.shape)
print("Test :", test_df.shape)


In [ ]:
#Separate Feature Matrix and Target Col
target_col = "default_flag"

X_train = train_df.drop(columns=target_col)
y_train = train_df[target_col]

X_test = test_df.drop(columns=target_col)
y_test = test_df[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

In [ ]:
# Load preprocessing pipeline (non-XGB)
xgb_preprocessor = joblib.load("../data/modeling/xg_preprocessor.pkl")
print("Loaded XGB preprocessor")

X_train_xgb_full_array = xgb_preprocessor.fit_transform(X_train)
xgb_feature_names_full = xgb_preprocessor.get_feature_names_out()

X_train_xgb_full = pd.DataFrame(
    data=X_train_xgb_full_array,
    columns=xgb_feature_names_full,
    index=X_train.index,
)

# ---- TRANSFORM FULL TEST ----
X_test_xgb_full_array = xgb_preprocessor.transform(X_test)

X_test_xgb_full = pd.DataFrame(
    data=X_test_xgb_full_array,
    columns=xgb_feature_names_full,
    index=X_test.index,
)

print("=== FULL XGB Preprocessing Complete ===")
print("Train shape:", X_train_xgb_full.shape)
print("Test shape :", X_test_xgb_full.shape)

In [ ]:
#Create TimesSeriesSplit and Evaluation Metric
timesplit = TimeSeriesSplit(n_splits=3)
f2_scorer = make_scorer(fbeta_score, beta=2)

In [ ]:
xgb_param_grid = {
    "max_depth":  [1, 2, 3, 5],
    "reg_alpha":  [1, 5, 8, 10, 20],
    "reg_lambda": [500, 1000, 1500, 2000],
}

In [ ]:
seeds = [42, 84, 126]

out_dir = "../results/xgb"

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos
print(f"XGB scale_pos_weight ≈ {scale_pos_weight:.2f} (neg={neg}, pos={pos})")

xgb_results_summary = []

for seed in seeds:
    print(f"\n================ XGB GRID SEARCH (seed={seed}) ================")

    base_xgb = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=2,
        scale_pos_weight=scale_pos_weight,
    )

    xgb_grid = GridSearchCV(
        estimator=base_xgb,
        param_grid=xgb_param_grid,
        scoring=f2_scorer,
        cv=timesplit,
        n_jobs=2,
        verbose=0,
        refit=True,
        return_train_score=False,
    )

    xgb_grid.fit(X_train_xgb_full, y_train)

    best_params = xgb_grid.best_params_
    best_cv_f2 = xgb_grid.best_score_
    best_model = xgb_grid.best_estimator_

    print(f"[seed={seed}] Best params: {best_params}")
    print(f"[seed={seed}] Best CV F2: {best_cv_f2:.6f}")

    cv_path = os.path.join(out_dir, f"xgb_seed{seed}_gridcv.joblib")
    model_path = os.path.join(out_dir, f"xgb_seed{seed}_model.joblib")
    joblib.dump(xgb_grid, cv_path)
    joblib.dump(best_model, model_path)
    print(f"[seed={seed}] Saved GridSearchCV → {cv_path}")
    print(f"[seed={seed}] Saved best model  → {model_path}")

    y_test_proba = best_model.predict_proba(X_test_xgb_full)[:, 1]
    y_test_pred = (y_test_proba >= 0.5).astype(int)

    test_f2 = fbeta_score(y_test, y_test_pred, beta=2)
    cm = confusion_matrix(y_test, y_test_pred)

    print(f"[seed={seed}] TEST F2: {test_f2:.6f}")
    print(f"[seed={seed}] Confusion matrix:\n{cm}")

    xgb_results_summary.append({
        "seed": seed,
        "cv_f2": float(best_cv_f2),
        "test_f2": float(test_f2),
        "best_params": best_params,
        "gridcv_path": cv_path,
        "model_path": model_path,
    })

summary_df = pd.DataFrame(xgb_results_summary)
print("\n================ XGB SUMMARY ACROSS SEEDS ================")
display(summary_df[["seed", "cv_f2", "test_f2", "best_params"]])
